# Tinh chỉnh 8 mô hình phân loại trên 5 fold (GSE68465, TCGA-LUAD, TCGA-LUSC)

Mô hình: **KNN, Gaussian NB, Logistic Regression, Decision Tree, Random Forest,
Linear SVM, RBF SVM, MLP**.

Notebook này dùng 6 file zip do `lung_download_label_5fold.ipynb` tạo ra
(`<bo>_h5.zip`, thư mục `MyDrive/KLTN/data/folds`). Chạy notebook đó trước nếu
chưa có zip.

1. **Dò lưới tham số** cho từng mô hình trên cả 5 fold của 3 bộ. Cấu hình được
   **chọn theo macro-F1 trung bình trên val**; test chỉ dùng để báo cáo, không
   dùng để chọn (tránh rò rỉ). Kết quả ghi dần vào CSV trên Drive — Colab bị ngắt
   thì chạy lại cell, cấu hình đã xong sẽ được bỏ qua.
2. **So sánh** cấu hình mặc định và cấu hình tốt nhất của từng mô hình, xếp hạng
   mô hình cho mỗi bộ.
3. **Chẩn đoán**: mỗi lớp được dự đoán bao nhiêu lần — có lớp nào không bao giờ
   được dự đoán không.
4. `classification_report` gộp trên test của 5 fold (mỗi mẫu làm test đúng 1 lần).

Dữ liệu đưa thẳng vào mô hình (giá trị log2 biểu hiện gen), **không chuẩn hoá /
không StandardScaler**. Các mô hình hỗ trợ `class_weight` được thử thêm
`class_weight="balanced"` do dữ liệu lệch lớp (KNN, Gaussian NB, MLP không hỗ trợ).

## 1. Thiết lập

In [ ]:
import os, sys, json, time, zipfile, itertools, warnings
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             balanced_accuracy_score, accuracy_score, recall_score)

warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDS_DIR = "/content/drive/MyDrive/KLTN/data/folds"            # noi co 6 file zip
    RESULT_DIR = "/content/drive/MyDrive/KLTN/results/model_tuning"  # ket qua (luu tren Drive)
    LOCAL_DIR = "/content/folds_local"                               # giai nen ra dia Colab cho nhanh
else:
    FOLDS_DIR = os.path.abspath(os.path.join("kltn_data", "folds"))
    RESULT_DIR = os.path.abspath(os.path.join("kltn_data", "model_tuning"))
    LOCAL_DIR = os.path.abspath(os.path.join("kltn_data", "folds_local"))
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

DATASET_NAMES = ["gse68465", "tcga_luad", "tcga_lusc"]
N_FOLDS = 5
SEED = 42
STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}

print("Colab:", IN_COLAB, "| so CPU:", os.cpu_count())
print("Zip  :", FOLDS_DIR)
missing = [n for n in DATASET_NAMES if not os.path.exists(os.path.join(FOLDS_DIR, f"{n}_h5.zip"))]
assert not missing, f"Chua co zip cho {missing} -- chay lung_download_label_5fold.ipynb truoc"

In [ ]:
%pip install -q tables

## 2. Nạp 5 fold của cả 3 bộ vào RAM

Giải nén từ Drive ra đĩa Colab (đọc nhanh hơn), chuyển sang `float32`.

In [ ]:
def load_dataset(name):
    zip_path = os.path.join(FOLDS_DIR, f"{name}_h5.zip")
    folds = {}
    with zipfile.ZipFile(zip_path) as zf:
        for r in range(1, N_FOLDS + 1):
            fold = {}
            for split in ["train", "val", "test"]:
                path = zf.extract(f"{name}/fold_{r}/{split}.h5", LOCAL_DIR)
                df = pd.read_hdf(path, key="data")
                fold[split] = (df.drop(columns="label").values.astype(np.float32),
                               df["label"].values.astype(int), df.index.values)
            folds[r] = fold
    return folds


DATA = {}
for name in DATASET_NAMES:
    t0 = time.time()
    DATA[name] = load_dataset(name)
    Xtr, ytr, _ = DATA[name][1]["train"]
    counts = pd.Series(ytr).map(STAGE_NAMES).value_counts().to_dict()
    print(f"{name:10s} fold1 train {Xtr.shape} | khoang gia tri [{Xtr.min():.1f}, {Xtr.max():.1f}] "
          f"| lop train: {counts} | {time.time() - t0:.0f}s")

## 3. Mô hình và lưới tham số

`DEFAULTS` là cấu hình mặc định (gần với mặc định của sklearn) để so sánh;
`GRIDS` là lưới dò. Cấu hình mặc định luôn được chạy kèm trong lưới.
`MODELS_TO_RUN` chọn mô hình cần chạy; `QUICK = True` dùng lưới nhỏ để thử nhanh.

Vì không chuẩn hoá, `gamma` của RBF SVM được dò quanh `"scale"`
(= 1 / (số gene × phương sai của X), cỡ 1e-5 với dữ liệu log2 ~12.500–20.000 gene).
Linear SVM dùng `SVC(kernel="linear")` thay cho `LinearSVC`: trên dữ liệu không
chuẩn hoá `LinearSVC` hội tụ rất chậm (chạy thử: hơn 6 phút cho 1 cấu hình), còn
`SVC` với < 500 mẫu chỉ mất vài giây. Random Forest và MLP chậm nhất.

In [ ]:
QUICK = False
MODELS_TO_RUN = ["knn", "gaussian_nb", "logistic_regression", "decision_tree",
                 "random_forest", "linear_svm", "rbf_svm", "mlp"]


def build_model(model, params):
    p = dict(params)
    if model == "knn":
        return KNeighborsClassifier(n_jobs=-1, **p)
    if model == "gaussian_nb":
        return GaussianNB(**p)
    if model == "logistic_regression":
        return LogisticRegression(max_iter=5000, random_state=SEED, **p)
    if model == "decision_tree":
        return DecisionTreeClassifier(random_state=SEED, **p)
    if model == "random_forest":
        return RandomForestClassifier(n_jobs=-1, random_state=SEED, **p)
    if model == "linear_svm":
        # SVC kernel tuyen tinh (libsvm): voi < 500 mau nhanh hon LinearSVC nhieu tren du
        # lieu khong chuan hoa (LinearSVC hoi tu rat cham, co cau hinh mat vai phut)
        return SVC(kernel="linear", random_state=SEED, **p)
    if model == "rbf_svm":
        return SVC(kernel="rbf", random_state=SEED, **p)
    if model == "mlp":
        p["hidden_layer_sizes"] = tuple(p["hidden_layer_sizes"])
        return MLPClassifier(max_iter=500, random_state=SEED, **p)
    raise ValueError(model)


DEFAULTS = {
    "knn": {"n_neighbors": 5, "weights": "uniform", "metric": "minkowski"},
    "gaussian_nb": {"var_smoothing": 1e-9},
    "logistic_regression": {"C": 1.0, "class_weight": None},
    "decision_tree": {"max_depth": None, "min_samples_leaf": 1, "criterion": "gini", "class_weight": None},
    "random_forest": {"n_estimators": 500, "max_features": "sqrt", "min_samples_leaf": 1,
                      "class_weight": None, "max_samples": None},
    "linear_svm": {"C": 1.0, "class_weight": None},
    "rbf_svm": {"C": 1.0, "gamma": "scale", "class_weight": None},
    "mlp": {"hidden_layer_sizes": [100], "alpha": 1e-4},
}

if QUICK:
    GRIDS = {
        "knn": {"n_neighbors": [5, 11], "weights": ["distance"], "metric": ["minkowski", "cosine"]},
        "gaussian_nb": {"var_smoothing": [1e-9, 1e-3]},
        "logistic_regression": {"C": [0.01, 1.0], "class_weight": ["balanced"]},
        "decision_tree": {"max_depth": [None, 5], "min_samples_leaf": [5], "criterion": ["gini"],
                          "class_weight": ["balanced"]},
        "random_forest": {"n_estimators": [500], "max_features": ["sqrt"], "min_samples_leaf": [1, 10],
                          "class_weight": ["balanced"], "max_samples": [None]},
        "linear_svm": {"C": [1e-3, 1.0], "class_weight": ["balanced"]},
        "rbf_svm": {"C": [1.0, 10.0], "gamma": ["scale"], "class_weight": ["balanced"]},
        "mlp": {"hidden_layer_sizes": [[100]], "alpha": [1e-4, 1.0]},
    }
else:
    GRIDS = {
        "knn": {"n_neighbors": [3, 5, 7, 11, 15, 21], "weights": ["uniform", "distance"],
                "metric": ["minkowski", "cosine"]},                                    # 24
        "gaussian_nb": {"var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3, 1e-2, 1e-1]},         # 6
        "logistic_regression": {"C": [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0],
                                "class_weight": [None, "balanced"]},                   # 12
        "decision_tree": {"max_depth": [None, 3, 5, 10, 20], "min_samples_leaf": [1, 5, 10],
                          "criterion": ["gini", "entropy"],
                          "class_weight": [None, "balanced"]},                         # 60
        "random_forest": {"n_estimators": [500], "max_features": ["sqrt", "log2", 0.05],
                          "min_samples_leaf": [1, 5, 10], "class_weight": ["balanced"],
                          "max_samples": [None, 0.5]},                                 # 18
        "linear_svm": {"C": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
                       "class_weight": [None, "balanced"]},                            # 12
        "rbf_svm": {"C": [0.1, 1.0, 10.0, 100.0], "gamma": ["scale", 1e-6, 1e-5, 1e-4],
                    "class_weight": [None, "balanced"]},                               # 32
        "mlp": {"hidden_layer_sizes": [[64], [256], [256, 64]],
                "alpha": [1e-4, 1e-2, 1.0, 10.0]},                                     # 12
    }


def config_key(model, params):
    return json.dumps({"model": model, **params}, sort_keys=True)


CONFIGS = []   # (model, params)
for model in MODELS_TO_RUN:
    grid = GRIDS[model]
    cfgs = [dict(zip(grid, v)) for v in itertools.product(*grid.values())]
    if DEFAULTS[model] not in cfgs:
        cfgs = [DEFAULTS[model]] + cfgs
    CONFIGS += [(model, c) for c in cfgs]

print(pd.Series([m for m, _ in CONFIGS]).value_counts().reindex(MODELS_TO_RUN).to_string())
print(f"Tong: {len(CONFIGS)} cau hinh x {N_FOLDS} fold x {len(DATASET_NAMES)} bo "
      f"= {len(CONFIGS) * N_FOLDS * len(DATASET_NAMES)} lan huan luyen")

## 4. Hàm huấn luyện và đánh giá

In [ ]:
def run_config(name, model, params, keep_pred=False):
    # Huan luyen tren 5 fold -> list dong ket qua (moi fold x split)
    rows, preds = [], []
    all_labels = sorted(set(np.concatenate([DATA[name][1][s][1] for s in ["train", "val", "test"]])))
    for r, fold in DATA[name].items():
        Xtr, ytr, _ = fold["train"]
        t0 = time.time()
        est = build_model(model, params).fit(Xtr, ytr)
        fit_s = time.time() - t0
        for split in ["val", "test"]:
            X, y, ids = fold[split]
            p = est.predict(X)
            rec = recall_score(y, p, labels=all_labels, average=None, zero_division=0)
            row = {"dataset": name, "model": model, "fold": r, "split": split, "fit_s": fit_s,
                   "macro_f1": f1_score(y, p, average="macro", zero_division=0),
                   "balanced_acc": balanced_accuracy_score(y, p),
                   "accuracy": accuracy_score(y, p)}
            for lab, rc in zip(all_labels, rec):
                row[f"recall_{STAGE_NAMES[lab]}"] = rc
                row[f"n_pred_{STAGE_NAMES[lab]}"] = int((p == lab).sum())
                row[f"n_true_{STAGE_NAMES[lab]}"] = int((y == lab).sum())
            rows.append(row)
            if keep_pred:
                preds.append(pd.DataFrame({"sample_id": ids, "fold": r, "split": split,
                                           "y_true": y, "y_pred": p}))
    return (rows, pd.concat(preds)) if keep_pred else rows

## 5. Dò lưới tham số

Mỗi bộ ghi 1 file `grid_<bo>.csv` trên Drive (các bộ có số lớp khác nhau — GSE68465
không có stage IV — nên cột khác nhau). Chạy lại cell này sẽ tiếp tục từ chỗ dừng.

In [ ]:
def result_csv(name):
    return os.path.join(RESULT_DIR, f"grid_{name}.csv")


t_start = time.time()
for name in DATASET_NAMES:
    done = set()
    if os.path.exists(result_csv(name)):
        done = set(pd.read_csv(result_csv(name))["config"])
        print(f"{name}: tiep tuc, da co {len(done)} cau hinh trong {result_csv(name)}")
    todo = [(m, p) for m, p in CONFIGS if config_key(m, p) not in done]
    for i, (model, params) in enumerate(todo, 1):
        t0 = time.time()
        rows = run_config(name, model, params)
        key = config_key(model, params)
        for row in rows:
            row["config"] = key
        pd.DataFrame(rows).to_csv(result_csv(name), mode="a", index=False,
                                  header=not os.path.exists(result_csv(name)))
        val_f1 = np.mean([r["macro_f1"] for r in rows if r["split"] == "val"])
        print(f"{name} {i}/{len(todo)} {model} {params} -> val macro-F1 {val_f1:.3f} "
              f"({time.time() - t0:.0f}s, tong {(time.time() - t_start) / 60:.1f} phut)", flush=True)

## 6. Chọn cấu hình tốt nhất cho từng mô hình (theo val) và so sánh với mặc định

`macro_f1_test` là trung bình qua 5 fold (kèm độ lệch chuẩn). Cấu hình được chọn
chỉ dựa trên `macro_f1_val`.

In [ ]:
metrics = ["macro_f1", "balanced_acc", "accuracy"]
BEST, compare_rows = {}, []

for name in DATASET_NAMES:
    res = pd.read_csv(result_csv(name))
    res = res[res["config"].isin([config_key(m, p) for m, p in CONFIGS])]
    recall_cols = [c for c in res.columns if c.startswith("recall_")]
    a = res.groupby(["model", "config", "split"])[metrics + recall_cols].mean().unstack("split")
    a.columns = [f"{m}_{s}" for m, s in a.columns]
    a["macro_f1_test_std"] = res.groupby(["model", "config", "split"])["macro_f1"].std().unstack("split")["test"]
    a["fit_s"] = res.groupby(["model", "config"])["fit_s"].mean()

    for model in MODELS_TO_RUN:
        if model not in a.index.get_level_values("model"):
            continue
        am = a.loc[model].sort_values("macro_f1_val", ascending=False)
        best_key = am.index[0]
        BEST[(name, model)] = {k: v for k, v in json.loads(best_key).items() if k != "model"}
        for kind, key in [("mac_dinh", config_key(model, DEFAULTS[model])), ("tot_nhat", best_key)]:
            row = am.loc[key]
            compare_rows.append({"dataset": name, "model": model, "cau_hinh": kind,
                                 "macro_f1_val": row["macro_f1_val"],
                                 "macro_f1_test": row["macro_f1_test"],
                                 "macro_f1_test_std": row["macro_f1_test_std"],
                                 "balanced_acc_test": row["balanced_acc_test"],
                                 "accuracy_test": row["accuracy_test"],
                                 **{c: row[c] for c in am.columns if c.startswith("recall_") and c.endswith("_test")},
                                 "fit_s": row["fit_s"]})

compare = pd.DataFrame(compare_rows).set_index(["dataset", "model", "cau_hinh"])
for name in DATASET_NAMES:
    print(f"\n===== {name}: mac dinh vs tot nhat =====")
    display(compare.loc[name].dropna(axis=1, how="all").round(3))

print("\nCau hinh tot nhat (theo val macro-F1):")
for (name, model), cfg in BEST.items():
    print(f"  {name:10s} {model:20s} {cfg}")

In [ ]:
# Xep hang mo hinh (cau hinh tot nhat) cho moi bo, theo val macro-F1
best_only = compare.xs("tot_nhat", level="cau_hinh")
for name in DATASET_NAMES:
    b = best_only.loc[name].sort_values("macro_f1_val", ascending=False)
    print(f"\n===== {name}: xep hang mo hinh =====")
    display(b[["macro_f1_val", "macro_f1_test", "macro_f1_test_std",
               "balanced_acc_test", "accuracy_test"]].round(3))

## 7. Chẩn đoán: có lớp nào không bao giờ được dự đoán?

Gộp test của 5 fold: `<lop>_du_doan` = số mẫu mô hình dự đoán là lớp đó, so với
số mẫu thật `<lop>_that`. Cột `lop_khong_duoc_du_doan` liệt kê các lớp có
`du_doan = 0` — mô hình bỏ qua lớp đó và dồn về lớp đông nhất.

In [ ]:
diag_rows = []
for name in DATASET_NAMES:
    res = pd.read_csv(result_csv(name))
    res = res[res["split"] == "test"]
    stages = [s for s in STAGE_NAMES.values() if f"n_true_{s}" in res.columns]
    for model in MODELS_TO_RUN:
        if (name, model) not in BEST:
            continue
        for kind, params in [("mac_dinh", DEFAULTS[model]), ("tot_nhat", BEST[(name, model)])]:
            r = res[res["config"] == config_key(model, params)]
            row = {"dataset": name, "model": model, "cau_hinh": kind}
            for s in stages:
                row[f"{s}_that"] = int(r[f"n_true_{s}"].sum())
                row[f"{s}_du_doan"] = int(r[f"n_pred_{s}"].sum())
            row["lop_khong_duoc_du_doan"] = ", ".join(s for s in stages if r[f"n_pred_{s}"].sum() == 0) or "-"
            diag_rows.append(row)

diag = pd.DataFrame(diag_rows).set_index(["dataset", "model", "cau_hinh"])
for name in DATASET_NAMES:
    print(f"\n===== {name} (tong tren test cua 5 fold) =====")
    display(diag.loc[name].dropna(axis=1, how="all"))

## 8. Classification report gộp trên test của 5 fold

Với cấu hình tốt nhất của từng mô hình: mỗi mẫu làm test đúng 1 lần qua 5 fold,
nên gộp dự đoán test lại cho ra 1 báo cáo trên toàn bộ dữ liệu. Đặt
`REPORT_MODELS` để chỉ in một số mô hình.

In [ ]:
REPORT_MODELS = MODELS_TO_RUN

final_preds = []
for name in DATASET_NAMES:
    for model in REPORT_MODELS:
        if (name, model) not in BEST:
            continue
        params = BEST[(name, model)]
        _, pred = run_config(name, model, params, keep_pred=True)
        pred = pred[pred["split"] == "test"].assign(dataset=name, model=model)
        final_preds.append(pred)

        y, p = pred["y_true"].values, pred["y_pred"].values
        labels = sorted(set(y) | set(p))
        names = [STAGE_NAMES[l] for l in labels]
        print("=" * 70)
        print(f"{name} | {model} | {params}")
        print(classification_report(y, p, labels=labels, target_names=names, digits=3, zero_division=0))
        display(pd.DataFrame(confusion_matrix(y, p, labels=labels),
                             index=[f"that_{n}" for n in names],
                             columns=[f"du_doan_{n}" for n in names]))

pd.concat(final_preds).to_csv(os.path.join(RESULT_DIR, "test_predictions_best.csv"), index=False)
compare.to_csv(os.path.join(RESULT_DIR, "compare_default_vs_best.csv"))
diag.to_csv(os.path.join(RESULT_DIR, "class_prediction_counts.csv"))
with open(os.path.join(RESULT_DIR, "best_params.json"), "w") as f:
    json.dump({f"{n}/{m}": cfg for (n, m), cfg in BEST.items()}, f, indent=2)
print("Da luu ket qua vao", RESULT_DIR, ":", sorted(os.listdir(RESULT_DIR)))

## 9. (Tuỳ chọn) Ngắt kết nối Colab

Kết quả đã nằm trên Drive (`grid_<bo>.csv`, `compare_default_vs_best.csv`,
`class_prediction_counts.csv`, `best_params.json`, `test_predictions_best.csv`).

In [ ]:
if IN_COLAB:
    from google.colab import runtime
    runtime.unassign()